# Probe 4-module — Time & VRAM estimate (10K, r=64)

**Muc dich:** So sanh VRAM + sec/step khi chi dung 4 attention modules (khong MLP).  
Cung config r=64, gradient_checkpointing=True. Doi chieu voi `04a_probe_7mod.ipynb`.  
**Khong push HF.** Decision: neu 7mod OOM hoac cham qua thi dung 4mod cho v2.

| Param | Value |
|-------|-------|
| Train size | 10,000 |
| Epochs | 1 |
| LoRA r | 64 |
| Target modules | **4** (q/k/v/o attention only) |
| gradient_checkpointing | True |

In [ ]:
import os, re, sys, gc, json, time, math
import numpy as np
from tqdm import tqdm
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List

import torch
import bitsandbytes as bnb
import torch.nn as nn
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import login
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, PreTrainedTokenizerBase
from trl import SFTTrainer, SFTConfig

@dataclass
class DataCollatorForCompletionOnlyLM:
    response_template: List[int]
    tokenizer: PreTrainedTokenizerBase
    ignore_index: int = -100

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_ids_list = [torch.tensor(f["input_ids"], dtype=torch.long) for f in features]
        max_len = max(len(x) for x in input_ids_list)
        bs = len(input_ids_list)
        padded = torch.full((bs, max_len), self.tokenizer.pad_token_id, dtype=torch.long)
        attn   = torch.zeros((bs, max_len), dtype=torch.long)
        labels = torch.full((bs, max_len), self.ignore_index, dtype=torch.long)
        tpl, tpl_len = self.response_template, len(self.response_template)
        for i, ids in enumerate(input_ids_list):
            n = len(ids)
            padded[i, :n] = ids
            attn[i, :n]   = 1
            for j in range(n - tpl_len, -1, -1):
                if ids[j : j + tpl_len].tolist() == tpl:
                    labels[i, j + tpl_len : n] = ids[j + tpl_len : n]
                    break
        return {"input_ids": padded, "attention_mask": attn, "labels": labels}

NOTEBOOK_DIR = Path("__file__").parent if "__file__" in dir() else Path(".")
sys.path.insert(0, str(NOTEBOOK_DIR))
from utils.evaluator import compute_metrics
print("Imports OK")

In [ ]:
# Constants — chi khac 04a o LORA_TARGET_MODULES
BASE_MODEL      = "Qwen/Qwen3.5-4B-Base"
DATASET_NAME    = "SeanSunny/items_prompts_tv_3"
MAX_SEQ_LENGTH  = 192
MAX_NEW_TOKENS  = 4
QUESTION_PREFIX = "S\u1ea3n ph\u1ea9m n\u00e0y c\u00f3 gi\u00e1 bao nhi\u00eau ?\n"
PRICE_PREFIX    = "\n\nGi\u00e1 l\u00e0: "

PROBE_TRAIN_SIZE    = 10000
PROBE_EVAL_SIZE     = 50
PROBE_EPOCHS        = 1
LORA_R              = 64
LORA_ALPHA          = 128
LORA_DROPOUT        = 0.1
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]   # 4 modules — khac 04a
PER_DEVICE_BATCH    = 16
GRAD_ACCUM          = 4
LEARNING_RATE       = 2e-4
GRADIENT_CHECKPOINTING = True
SEED                = 42
PRED_CLAMP_MIN, PRED_CLAMP_MAX = 5, 1000
PARSE_REGEX = r"[-+]?\d*\.\d+|\d+"

assert torch.cuda.is_available()
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
load_dotenv(NOTEBOOK_DIR.parent / ".env")
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(HF_TOKEN)
    print("HF login OK")

In [ ]:
# Load model — torch_dtype=torch.bfloat16 fix conv1d bug
quant_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"EOS: {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=quant_config,
    device_map="auto", trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=GRADIENT_CHECKPOINTING)
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

In [ ]:
# Verify + apply LoRA 4 modules
linear_suffixes = set()
for name, module in model.named_modules():
    if isinstance(module, (nn.Linear, bnb.nn.Linear4bit)):
        linear_suffixes.add(name.split(".")[-1])
print("Linear suffixes:", sorted(linear_suffixes))

EXPECTED = {"q_proj", "k_proj", "v_proj", "o_proj"}
target_modules = LORA_TARGET_MODULES if EXPECTED.issubset(linear_suffixes) else "all-linear"
print(f"target_modules = {target_modules}")

model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=target_modules, bias="none", task_type="CAUSAL_LM",
))
model.print_trainable_parameters()
model.enable_input_require_grads()

In [ ]:
# Dataset + preprocess
ds = load_dataset(DATASET_NAME)
q_ids = tokenizer.encode(QUESTION_PREFIX, add_special_tokens=False)
p_ids = tokenizer.encode(PRICE_PREFIX, add_special_tokens=False)
TOKENS_FIXED = len(q_ids) + len(p_ids)
MAX_SUMMARY_TOKENS = MAX_SEQ_LENGTH - TOKENS_FIXED - MAX_NEW_TOKENS - 1 - 2

def preprocess(example):
    p = example["prompt"]
    summary = p[len(QUESTION_PREFIX):-len(PRICE_PREFIX)]
    ids = tokenizer.encode(summary, add_special_tokens=False)
    if len(ids) > MAX_SUMMARY_TOKENS:
        summary = tokenizer.decode(ids[:MAX_SUMMARY_TOKENS], skip_special_tokens=True).rstrip()
    return {"text": QUESTION_PREFIX + summary + PRICE_PREFIX + example["completion"] + "\n" + tokenizer.eos_token}

def tokenize_fn(example):
    return tokenizer(example["text"], truncation=True, max_length=MAX_SEQ_LENGTH, padding=False)

train_raw = ds["train"].shuffle(seed=SEED).select(range(PROBE_TRAIN_SIZE))
val_raw   = ds["val"].shuffle(seed=SEED).select(range(PROBE_EVAL_SIZE))

train_ds_tok = train_raw.map(preprocess, desc="Preprocess train").map(tokenize_fn, batched=False, remove_columns=["text"])
val_ds_tok   = val_raw.map(preprocess, desc="Preprocess val").map(tokenize_fn, batched=False, remove_columns=["text"])
print(f"train_ds_tok: {len(train_ds_tok):,} | val_ds_tok: {len(val_ds_tok):,}")

In [ ]:
# Collator + mask verify
response_template_ids = tokenizer.encode(PRICE_PREFIX, add_special_tokens=False)
collator = DataCollatorForCompletionOnlyLM(response_template=response_template_ids, tokenizer=tokenizer)

tok = tokenizer(preprocess(train_raw[0])["text"],
                return_tensors="pt", max_length=MAX_SEQ_LENGTH, truncation=True)
batch_out = collator([{"input_ids": tok["input_ids"][0].tolist(),
                       "attention_mask": tok["attention_mask"][0].tolist()}])
non_masked = batch_out["labels"][0][batch_out["labels"][0] != -100]
if len(non_masked) == 0:
    raise RuntimeError("Mask verify FAILED. Abort.")
print(f"Mask verify PASS: {tokenizer.decode(non_masked.tolist(), skip_special_tokens=False)!r}")

In [ ]:
# Train 10K / 1 epoch — do VRAM + time
torch.cuda.reset_peak_memory_stats()
t_start = time.time()

trainer = SFTTrainer(
    model=model, processing_class=tokenizer,
    train_dataset=train_ds_tok, eval_dataset=val_ds_tok,
    data_collator=collator,
    args=SFTConfig(
        output_dir=str(NOTEBOOK_DIR / "weights" / "probe_4mod"),
        num_train_epochs=PROBE_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BATCH,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        bf16=True,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=10,
        report_to="none",
        seed=SEED,
    ),
)
trainer.train()

t_elapsed = time.time() - t_start
vram_peak = torch.cuda.max_memory_allocated() / 1e9
steps_10k = math.ceil(PROBE_TRAIN_SIZE / (PER_DEVICE_BATCH * GRAD_ACCUM))
sec_per_step = t_elapsed / steps_10k

steps_full_v2 = math.ceil(85727 / (PER_DEVICE_BATCH * GRAD_ACCUM)) * 3
est_full_min = sec_per_step * steps_full_v2 / 60

print("\n" + "=" * 55)
print("PROBE 4-MODULE RESULTS")
print("=" * 55)
print(f"Train time (10K/1ep) : {t_elapsed/60:.1f} min")
print(f"VRAM peak            : {vram_peak:.2f} GB")
print(f"Sec/step             : {sec_per_step:.2f}s")
print(f"Steps (10K/1ep)      : {steps_10k}")
print(f"Steps (full v2)      : {steps_full_v2}  (85K x 3ep)")
print(f"Est. full v2 time    : {est_full_min:.0f} min ({est_full_min/60:.1f} hr)")
print("=" * 55)

log_history = trainer.state.log_history
train_losses = [(e["step"], e["loss"]) for e in log_history if "loss" in e and "eval_loss" not in e]
if train_losses:
    print(f"Train loss: {train_losses[0][1]:.4f} -> {train_losses[-1][1]:.4f}")

In [ ]:
# Quick generative eval 50 val
del trainer
gc.collect()
torch.cuda.empty_cache()
model.eval()

preds_vnd, trues_vnd = [], []
for item in tqdm(val_raw, desc="Quick eval"):
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(PARSE_REGEX, gen)
    pk = max(PRED_CLAMP_MIN, min(int(float(m.group())), PRED_CLAMP_MAX)) if m else 0
    preds_vnd.append(pk * 1000)
    trues_vnd.append(item["price_vnd_true"])

metrics = compute_metrics(np.array(trues_vnd, dtype=float), np.array(preds_vnd, dtype=float))
print(f"Quick eval (50 val, 1ep 10K): RMSLE={metrics['rmsle']:.4f}, MAE={metrics['mae']:,.0f}")
print(f"Zero preds: {preds_vnd.count(0)}")

gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## Decision guide sau khi chay ca 2 probe

| Ket qua 7mod | Ket qua 4mod | Quyet dinh v2 |
|---|---|---|
| VRAM < 23GB, time hop ly | — | Dung 7mod cho `04_train_v2.ipynb` |
| VRAM > 23GB hoac crash | — | Dung 4mod + tang GRAD_ACCUM neu can |
| Time 7mod >> 4mod (>50%) | Loss 7mod > 4mod sau 1ep | Can can nhac: 7mod may improve nhung rat cham |
| Time 7mod ~= 4mod | Loss 7mod < 4mod | Chac chan dung 7mod |